In [20]:
#!/usr/bin/env python

import ctypes as cts
import sys

import numpy as np
import time

from scipy.spatial.distance import directed_hausdorff



def main(*argv):

    rows0 = 42000
    cols0 = 3
    kernel_size = 3
    rows1 = 30000
    cols1 = 3
    mat0 = np.random.randn(rows0, cols0).astype(cts.c_double)
    mat1 = np.random.randn(rows1, cols1).astype(cts.c_double)
    
    
    start = time.time()
    mat_res,dealloc_array = haus(mat0, rows0, cols0, mat1, rows1, cols1)
    print("Johnny")
    print(max(mat_res))
    end = time.time()
    print(f"Time: {end - start}")
    
    dealloc_array(mat_res)
    
    print("\n")
    print("Scipy")
    start = time.time()
    scipy__dist = directed_hausdorff(mat0, mat1)
    print(scipy__dist)
    end = time.time()
    
    print(f"Time: {end - start}")
    start = time.time()
    #yasen = hausdorff_distance(mat0, mat1)
    print("\n")
    print("Yasen")
    #print(max(yasen))
    end = time.time()
    print(f"Yasen Time: {end - start}")
    
if __name__ == "__main__":
    print("Python {:s} {:03d}bit on {:s}\n".format(" ".join(elem.strip() for elem in sys.version.split("\n")),
                                                   64 if sys.maxsize > 0x100000000 else 32, sys.platform))
    rc = main(*sys.argv[1:])
    #sys.exit(rc)

Python 3.8.5 (default, Sep  3 2020, 21:29:08) [MSC v.1916 64 bit (AMD64)] 064bit on win32

Johnny
[1.3197745]
Time: 11.653615713119507


Scipy
(1.3197745010192299, 14542, 14158)
Time: 0.2270510196685791


Yasen
Yasen Time: 0.0


In [19]:

DLL_NAME = "./haus_40.{:s}".format("dll" if sys.platform[:3].lower() == "win" else "so")

def np_mat_type(rows, cols, element_type=float):
    return np.ctypeslib.ndpointer(dtype=element_type, shape=(rows, cols), flags="C_CONTIGUOUS")


def haus(mat0, rows0, cols0, mat1, rows1, cols1):

    #mat0 = np.array([[1,2,3],[2,2,2],[3,3,3],[1,4,3],[5,2,3]]).astype(cts.c_double)
    #mat1 = np.array([[4,4,4],[5,4,4],[6,4,4],[7,4,4],[8,4,4],[9,4,4]]).astype(cts.c_double)
    
    
    dll = cts.CDLL(DLL_NAME)
    matrix_func = dll.matrixFunc
    matrix_func.argtypes = (
        np_mat_type(rows0, cols0), cts.c_size_t, cts.c_size_t,
        np_mat_type(rows1, cols1), cts.c_size_t, cts.c_size_t)
    
    #all point
    matrix_func.restype = np_mat_type(rows0, 1)
    #one point
    #matrix_func.restype = np_mat_type(1, 1)
    
    
    dealloc_array = dll.deallocArray
    #all point
    dealloc_array.argtypes = (np_mat_type(rows0, 1),)
    #one point
    #dealloc_array.argtypes = (np_mat_type(1, 1),)
    
    dealloc_array.restype = None

    #print("mat0:")
    #print(mat0)
    #print("\nmat1:")
    #print(mat1)
    start = time.time()
    mat_res = matrix_func(mat0, rows0, cols0, mat1, rows1, cols1)
    return mat_res,dealloc_array
    #dealloc_array(mat_res)
    

In [5]:
from scipy.spatial.distance import directed_hausdorff
import numpy as np


rows0 = 12000
cols0 = 3
kernel_size = 3
rows1 = 30000
cols1 = 3

mat0 = np.random.randn(rows0, cols0)

mat1 = np.random.randn(rows1, cols1)
directed_hausdorff(mat0, mat1)

(1.4516636653256352, 10298, 14907)

In [6]:
def hausdorff_distance(P,Q):
    start_time=time.time()
    #inputs P and Q are arrays of vert coordinates

    dist = np.zeros((P.shape[0], 1))

    for p in range(P.shape[0]):

        # Calculate the minimum distance from points in P to Q

        minP = np.min(np.sum((P[p, :] - Q)**2, axis=1))

        dist[p, 0] = minP



    hd = np.sqrt(dist)
    end_time = time.time()
    return hd


In [6]:
import numpy as np
rows0 = 10
cols0 = 9
mat0 = np.random.randn(rows0, cols0)
r,c=mat0.shape
r

10